In [1]:
# Use a GPU runtime first: Runtime -> Change runtime type -> GPU

!pip -q install --upgrade pip
!pip -q install "torch==2.2.0" "torchvision==0.17.0" "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10"
!pip -q install draccus wandb sentencepiece accelerate einops pillow imageio imageio-ffmpeg
!pip -q install tensorflow

print("Install step finished. If Colab asks for restart, restart runtime, then continue.")

Install step finished. If Colab asks for restart, restart runtime, then continue.


In [2]:
%cd /content

!rm -rf /content/openvla
!rm -rf /content/LIBERO

!git clone https://github.com/openvla/openvla.git
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git

# Install LIBERO repo code only
!pip -q install -e /content/LIBERO --no-deps

# Install OpenVLA repo code only
!pip -q install -e /content/openvla --no-deps

# Install extra packages needed for LIBERO evaluation
!pip -q install "imageio[ffmpeg]" robosuite==1.4.1 bddl easydict cloudpickle gym
!pip -q install huggingface_hub json-numpy jsonlines matplotlib peft==0.11.1 rich torchaudio==2.2.0

print("Cell 2 finished.")

/content
Cloning into 'openvla'...
remote: Enumerating objects: 489, done.
remote: Total 489 (delta 0), reused 0 (delta 0), pack-reused 489 (from 1)
Receiving objects: 100% (489/489), 346.99 KiB | 1.35 MiB/s, done.
Resolving deltas: 100% (194/194), done.
Cloning into 'LIBERO'...
remote: Enumerating objects: 1788, done.
remote: Counting objects: 100% (455/455), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 1788 (delta 290), reused 290 (delta 290), pack-reused 1333 (from 1)
Receiving objects: 100% (1788/1788), 315.98 MiB | 27.29 MiB/s, done.
Resolving deltas: 100% (755/755), done.
Updating files: 100% (1116/1116), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for libero (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_

In [3]:
# Only fix draccus. Keep the sentencepiece version that Colab already has.
!pip -q install draccus==0.8.0

import draccus, sentencepiece
print("draccus:", draccus.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("Cell 2.5 finished.")

draccus: 0.3.1
sentencepiece: 0.2.1
Cell 2.5 finished.


In [4]:
from pathlib import Path

p = Path("/content/openvla/experiments/robot/openvla_utils.py")
txt = p.read_text()

txt = txt.replace(
    'print("[*] Loading in BF16 with Flash-Attention Enabled")',
    'print("[*] Loading model with eager attention fallback")'
)

txt = txt.replace(
    'attn_implementation="flash_attention_2",',
    'attn_implementation="eager",'
)

p.write_text(txt)
print("Patched openvla_utils.py")

Patched openvla_utils.py


In [5]:
%%writefile /content/openvla/experiments/robot/libero/silentvla_probe_eval.py
import os
import sys
import json
import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union

sys.path.append("/content/LIBERO")
sys.path.append("/content/openvla")
sys.path.append("../..")

import numpy as np
import tqdm

from libero.libero import benchmark
from experiments.robot.libero.libero_utils import (
    get_libero_dummy_action,
    get_libero_env,
    get_libero_image,
    quat2axisangle,
    save_rollout_video,
)
from experiments.robot.openvla_utils import get_processor, get_vla_action
from experiments.robot.robot_utils import (
    DATE_TIME,
    get_image_resize_size,
    get_model,
    invert_gripper_action,
    normalize_gripper_action,
    set_seed_everywhere,
)

@dataclass
class GenerateConfig:
    model_family: str = "openvla"
    pretrained_checkpoint: Union[str, Path] = "openvla/openvla-7b-finetuned-libero-spatial"
    load_in_8bit: bool = False
    load_in_4bit: bool = False
    center_crop: bool = True
    task_suite_name: str = "libero_spatial"
    num_steps_wait: int = 10
    num_trials_per_task: int = 1
    max_tasks: int = 1
    neutral_instruction: str = "perform the task"
    repeat_eps: float = 0.02
    run_id_note: Optional[str] = "probe_smoke"
    local_log_dir: str = "./experiments/logs"
    seed: int = 7

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_family", type=str, default="openvla")
    parser.add_argument("--pretrained_checkpoint", type=str, default="openvla/openvla-7b-finetuned-libero-spatial")
    parser.add_argument("--load_in_8bit", action="store_true")
    parser.add_argument("--load_in_4bit", action="store_true")
    parser.add_argument("--center_crop", type=lambda x: str(x).lower() == "true", default=True)
    parser.add_argument("--task_suite_name", type=str, default="libero_spatial")
    parser.add_argument("--num_steps_wait", type=int, default=10)
    parser.add_argument("--num_trials_per_task", type=int, default=1)
    parser.add_argument("--max_tasks", type=int, default=1)
    parser.add_argument("--neutral_instruction", type=str, default="perform the task")
    parser.add_argument("--repeat_eps", type=float, default=0.02)
    parser.add_argument("--run_id_note", type=str, default="probe_smoke")
    parser.add_argument("--local_log_dir", type=str, default="./experiments/logs")
    parser.add_argument("--seed", type=int, default=7)
    args = parser.parse_args()
    return GenerateConfig(
        model_family=args.model_family,
        pretrained_checkpoint=args.pretrained_checkpoint,
        load_in_8bit=args.load_in_8bit,
        load_in_4bit=args.load_in_4bit,
        center_crop=args.center_crop,
        task_suite_name=args.task_suite_name,
        num_steps_wait=args.num_steps_wait,
        num_trials_per_task=args.num_trials_per_task,
        max_tasks=args.max_tasks,
        neutral_instruction=args.neutral_instruction,
        repeat_eps=args.repeat_eps,
        run_id_note=args.run_id_note,
        local_log_dir=args.local_log_dir,
        seed=args.seed,
    )

def to_float(x):
    return float(x) if x is not None else None

def to_list(x):
    return [float(v) for v in x]

def postprocess_action(cfg, action):
    action = np.array(action, dtype=np.float32).copy()
    action = normalize_gripper_action(action, binarize=True)
    if cfg.model_family == "openvla":
        action = invert_gripper_action(action)
    return action

def build_observation(obs, img):
    return {
        "full_image": img,
        "state": np.concatenate(
            (
                obs["robot0_eef_pos"],
                quat2axisangle(obs["robot0_eef_quat"]),
                obs["robot0_gripper_qpos"],
            )
        ),
    }

def predict_action(cfg, model, processor, observation, task_label):
    action = get_vla_action(
        model,
        processor,
        cfg.pretrained_checkpoint,
        observation,
        task_label,
        cfg.task_suite_name,
        center_crop=cfg.center_crop,
    )
    return postprocess_action(cfg, action)

def eval_probe(cfg: GenerateConfig) -> None:
    assert not (cfg.load_in_8bit and cfg.load_in_4bit), "Cannot use both 8bit and 4bit"
    set_seed_everywhere(cfg.seed)

    model = get_model(cfg)
    processor = get_processor(cfg)
    resize_size = get_image_resize_size(cfg)

    run_id = f"SILENTVLA-{cfg.task_suite_name}-{DATE_TIME}"
    if cfg.run_id_note:
        run_id += f"--{cfg.run_id_note}"

    root_dir = os.path.join(cfg.local_log_dir, "silentvla_probe", run_id)
    os.makedirs(root_dir, exist_ok=True)

    summary_path = os.path.join(root_dir, "episode_summaries.jsonl")
    txt_log_path = os.path.join(root_dir, "run_log.txt")

    txt_log = open(txt_log_path, "w", encoding="utf-8")
    summary_file = open(summary_path, "a", encoding="utf-8")

    benchmark_dict = benchmark.get_benchmark_dict()
    task_suite = benchmark_dict[cfg.task_suite_name]()
    num_tasks = min(task_suite.n_tasks, cfg.max_tasks)

    total_eps = 0
    total_success = 0

    for task_id in tqdm.tqdm(range(num_tasks), desc="tasks"):
        task = task_suite.get_task(task_id)
        initial_states = task_suite.get_task_init_states(task_id)
        env, task_description = get_libero_env(task, cfg.model_family, resolution=256)

        for episode_idx in tqdm.tqdm(range(cfg.num_trials_per_task), desc=f"task_{task_id}", leave=False):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])

            if cfg.task_suite_name == "libero_spatial":
                max_steps = 220
            elif cfg.task_suite_name == "libero_object":
                max_steps = 280
            elif cfg.task_suite_name == "libero_goal":
                max_steps = 300
            elif cfg.task_suite_name == "libero_10":
                max_steps = 520
            elif cfg.task_suite_name == "libero_90":
                max_steps = 400
            else:
                raise ValueError(f"Unexpected suite {cfg.task_suite_name}")

            episode_dir = os.path.join(root_dir, f"task_{task_id:02d}_ep_{episode_idx:03d}")
            os.makedirs(episode_dir, exist_ok=True)
            step_log_path = os.path.join(episode_dir, "steps.jsonl")

            prev_action = None
            repetition_run = 0
            max_repetition_run = 0
            replay_images = []
            step_records = []
            success = False
            t = 0

            txt_log.write(f"\nTASK {task_id} | EP {episode_idx} | {task_description}\n")
            txt_log.flush()

            while t < max_steps + cfg.num_steps_wait:
                try:
                    if t < cfg.num_steps_wait:
                        obs, reward, done, info = env.step(get_libero_dummy_action(cfg.model_family))
                        t += 1
                        continue

                    img = get_libero_image(obs, resize_size)
                    replay_images.append(img)
                    observation = build_observation(obs, img)

                    action = predict_action(cfg, model, processor, observation, task_description)
                    neutral_action = predict_action(cfg, model, processor, observation, cfg.neutral_instruction)

                    action_norm = float(np.linalg.norm(action))
                    neutral_gap = float(np.linalg.norm(action - neutral_action))

                    if prev_action is None:
                        action_delta = 0.0
                        repeated = 0
                        repetition_run = 0
                    else:
                        action_delta = float(np.linalg.norm(action - prev_action))
                        repeated = int(action_delta < cfg.repeat_eps)
                        repetition_run = repetition_run + 1 if repeated else 0

                    max_repetition_run = max(max_repetition_run, repetition_run)

                    obs, reward, done, info = env.step(action.tolist())

                    record = {
                        "task_id": task_id,
                        "episode_idx": episode_idx,
                        "step": int(t),
                        "task_description": task_description,
                        "neutral_instruction": cfg.neutral_instruction,
                        "action": to_list(action),
                        "neutral_action": to_list(neutral_action),
                        "action_norm": action_norm,
                        "action_delta": action_delta,
                        "neutral_gap": neutral_gap,
                        "repeated": repeated,
                        "repetition_run": int(repetition_run),
                        "reward": float(reward),
                        "done_after_step": bool(done),
                    }
                    step_records.append(record)

                    prev_action = action.copy()
                    t += 1

                    if done:
                        success = True
                        break

                except Exception as e:
                    txt_log.write(f"EXCEPTION: {repr(e)}\n")
                    txt_log.flush()
                    break

            with open(step_log_path, "w", encoding="utf-8") as f:
                for r in step_records:
                    f.write(json.dumps(r) + "\n")

            if len(step_records) > 0:
                mean_action_delta = float(np.mean([r["action_delta"] for r in step_records]))
                mean_neutral_gap = float(np.mean([r["neutral_gap"] for r in step_records]))
                mean_action_norm = float(np.mean([r["action_norm"] for r in step_records]))
            else:
                mean_action_delta = None
                mean_neutral_gap = None
                mean_action_norm = None

            ep_summary = {
                "task_id": task_id,
                "episode_idx": episode_idx,
                "task_description": task_description,
                "success": bool(success),
                "steps_recorded": len(step_records),
                "mean_action_delta": to_float(mean_action_delta),
                "mean_neutral_gap": to_float(mean_neutral_gap),
                "mean_action_norm": to_float(mean_action_norm),
                "max_repetition_run": int(max_repetition_run),
                "step_log_path": step_log_path,
            }

            summary_file.write(json.dumps(ep_summary) + "\n")
            summary_file.flush()

            total_eps += 1
            if success:
                total_success += 1

            try:
                save_rollout_video(
                    replay_images,
                    total_eps,
                    success=success,
                    task_description=task_description,
                    log_file=txt_log,
                )
            except Exception as e:
                txt_log.write(f"VIDEO SAVE SKIPPED: {repr(e)}\n")

            txt_log.write(
                f"EP DONE | success={success} | total_success={total_success}/{total_eps} "
                f"({100.0 * total_success / max(total_eps,1):.1f}%)\n"
            )
            txt_log.flush()

    txt_log.close()
    summary_file.close()
    print(f"Finished. Logs saved in: {root_dir}")

if __name__ == "__main__":
    cfg = parse_args()
    eval_probe(cfg)

Writing /content/openvla/experiments/robot/libero/silentvla_probe_eval.py


In [7]:
!pip -q install "numpy<2"

import numpy
print("numpy:", numpy.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openvla 0.0.3 requires dlimp @ git+https://github.com/moojink/dlimp_openvla, which is not installed.
openvla 0.0.3 requires tensorflow_graphics==2021.12.3, which is not installed.
openvla 0.0.3 requires sentencepiece==0.1.99, but you have sentencepiece 0.2.1 which is incompatible.
openvla 0.0.3 requires tensorflow==2.15.0, but you have tensorflow 2.19.0 which is incompatible.
openvla 0.0.3 requires tensorflow_datasets==4.9.3, but you have tensorflow-datasets 4.9.9 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.
tobler

In [10]:
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)

numpy: 1.26.4
torch: 2.2.0+cu121


In [11]:
import sys
sys.path.append("/content/LIBERO")
sys.path.append("/content/openvla")

import libero
from libero.libero import benchmark

print("LIBERO import works.")

LIBERO import works.


In [12]:
%cd /content/openvla

!python experiments/robot/libero/silentvla_probe_eval.py \
  --model_family openvla \
  --pretrained_checkpoint openvla/openvla-7b-finetuned-libero-spatial \
  --task_suite_name libero_spatial \
  --center_crop True \
  --num_trials_per_task 1 \
  --max_tasks 1 \
  --run_id_note first_smoke

/content/openvla
2026-04-10 09:20:42.665526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775812842.685905   16036 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775812842.692568   16036 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775812842.709792   16036 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775812842.709832   16036 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775812842.709836   16036 computation_placer.cc:177] compu

In [13]:
!pip -q install git+https://github.com/moojink/dlimp_openvla

import dlimp
print("dlimp import works.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from dlimp) (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0


ModuleNotFoundError: No module named 'dlimp'

In [9]:
import os, glob, json, pandas as pd

runs = sorted(glob.glob("/content/openvla/experiments/logs/silentvla_probe/*"))
print("runs:", runs[-3:])

latest = runs[-1]
print("latest:", latest)

summary_path = os.path.join(latest, "episode_summaries.jsonl")
rows = [json.loads(line) for line in open(summary_path, "r", encoding="utf-8")]
df = pd.DataFrame(rows)
print(df)

step_logs = glob.glob(os.path.join(latest, "task_*", "steps.jsonl"))
print("num step logs:", len(step_logs))
if step_logs:
    sample = [json.loads(line) for line in open(step_logs[0], "r", encoding="utf-8")]
    print("sample step count:", len(sample))
    print("first step record:", sample[0] if sample else "empty")

runs: []


IndexError: list index out of range